In [57]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from copy import deepcopy
import multiprocessing as mp
import logging
import random
from itertools import cycle
import os
from functools import partial

import mlflow
import matplotlib.pyplot as plt
from networkx import number_weakly_connected_components
import torch
import numpy as np
from dotenv import load_dotenv

from halcyon.ModelRunner import run_training_job, ModelRunner
from halcyon.Simulators import NoiseLevel, SimulationResult

from halcyon.Models import PhoenixAddAndProdLayers, PyTorchODE, HybridModel
from halcyon.Models.BaseLineModels import AlwaysDerivativeZero
from halcyon.Models.Losses import MSELossPlusMachineLearningL2


seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Check if MLFlow connects correctly

In [58]:
print(os.getenv("MLFLOW_TRACKING_URI"))

https://mlflow.cropresilience.org


In [59]:
[f"{experiment.name=}" for experiment in mlflow.search_experiments()]
mlflow.set_experiment("Halcyon sandbox experiment")
mlflow.set_experiment_tag("Tool name", "HALCYON") 

# Input data

## Downloading data from MLFlow

In [60]:
sim_run_id_to_use = "8f557eb8aadd45fdba24175cc74672c3"  
run = mlflow.get_run(sim_run_id_to_use)

params = run.data.params
print(params)

{'nr_genes': '20', 'sparsity': '0.1', 'num_modules': '2', 'noise_level': 'LOW', 'seed': '174'}


In [61]:
ml_flow_download_path = mlflow.artifacts.download_artifacts(run_id=sim_run_id_to_use, artifact_path="dataset")
sim_output = SimulationResult.load_from_pkl(Path(ml_flow_download_path) / "simulation_result.pkl")

2026-03-12 12:43:55,619 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,638 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,643 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,654 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,668 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,681 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,694 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:43:55,709 - WARNING - Connection p

## Convert sim output into datasets for use with the trainer object

In [62]:
# From sim output to datasets
single_ko_dataset, double_ko_dataset, time_scaling_factor = sim_output.create_datasets()
train_ko_dataset, validation_ko_dataset = single_ko_dataset.split_train_test(test_fraction=0.25)

train_dataset_name = "train_single_ko"
val_dataset_name ="val_single_ko"
test_dataset_name = "double_ko"
dataset_dict = {
    train_dataset_name: train_ko_dataset,
    val_dataset_name: validation_ko_dataset,
    test_dataset_name: double_ko_dataset,}

In [63]:
for name, dataset in dataset_dict.items():
    print(f"Length {name}: {len(dataset)}")

Length train_single_ko: 15
Length val_single_ko: 5
Length double_ko: 20


# Trainers (with different models) creation

In [64]:
train_jobs : list[ModelRunner] = []
device = cycle(['cuda:0', 'cuda:1'])
batch_size = 16
nr_epochs = 5

## ML model

In [65]:
for hidden_size in [10]:
    for do_only_pos in [True]:        
        ml_model = PhoenixAddAndProdLayers(
                    modelled_gene_labels=single_ko_dataset.genes,
                    hidden_size=hidden_size,
                    condition_names=single_ko_dataset.all_environments,
                    synthesis_only_positive=do_only_pos,
                    device=next(device)
            )

        # my_loss = MSELossPlusMachineLearningL2(lambda_param=0.5)
        model_runner = ModelRunner(
            model = ml_model,
            batch_size=batch_size,
            nr_epochs=nr_epochs,
            optimizer_kwargs = {'lr': 0.01}
        )
        train_jobs.append(model_runner)

## Baseline model (always derivative of 0)

In [66]:
# baseline_model = AlwaysDerivativeZero(
#     modelled_gene_labels=single_ko_dataset.genes,
#     condition_names=single_ko_dataset.all_environments
#     )

# baseline_trainer = ModelRunner(
#     baseline_model,
#     batch_size=batch_size,
#     nr_epochs=0,
# )
# train_jobs.append(baseline_trainer)

In [67]:
for job in train_jobs:
    print(job)
    print()

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=10', 'synthesis_only_positive=True'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 nr_epochs=5,
 batch_size=16
)



In [68]:
train_jobs[0]

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=10', 'synthesis_only_positive=True'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 nr_epochs=5,
 batch_size=16
)

In [69]:
train_jobs[0].train_and_predict_for_mlflow(dataset_dict=dataset_dict, 
                                           train_data_key_in_dict=train_dataset_name,
                                           val_data_key_in_dict=val_dataset_name,
                                           test_data_key_in_dict=test_dataset_name,
                                           simulation_parent_id=sim_run_id_to_use,
                                           save_every_n_epochs=10)

2026-03-12 12:43:59,833 [MainProcess] Epoch 0, train_loss	0.17096802592277527
2026-03-12 12:44:00,278 [MainProcess] Epoch 0, val_loss	0.1358623057603836
2026-03-12 12:44:01,360 [MainProcess] Epoch 0, test_loss	0.18694999814033508
2026-03-12 12:44:02,241 [MainProcess] Epoch 1, train_loss	0.1478358805179596
2026-03-12 12:44:02,684 [MainProcess] Epoch 1, val_loss	0.11728674918413162
2026-03-12 12:44:03,764 [MainProcess] Epoch 1, test_loss	0.1692150980234146
2026-03-12 12:44:04,646 [MainProcess] Epoch 2, train_loss	0.12868191301822662
2026-03-12 12:44:05,083 [MainProcess] Epoch 2, val_loss	0.10219154506921768
2026-03-12 12:44:06,182 [MainProcess] Epoch 2, test_loss	0.15434704720973969
2026-03-12 12:44:07,058 [MainProcess] Epoch 3, train_loss	0.11293233931064606
2026-03-12 12:44:07,514 [MainProcess] Epoch 3, val_loss	0.09017370641231537
2026-03-12 12:44:08,589 [MainProcess] Epoch 3, test_loss	0.14211776852607727
2026-03-12 12:44:09,472 [MainProcess] Epoch 4, train_loss	0.10017216950654984
2

🏃 View run PhoenixAddAndProdLayers at: https://mlflow.cropresilience.org/#/experiments/3/runs/7f4f07391d9d493a9ca208b6a11b1f56
🧪 View experiment at: https://mlflow.cropresilience.org/#/experiments/3


# Do training

In [70]:
# mp.set_start_method("spawn", force=True)
# with mp.Pool(processes=4) as pool:
#     pool.map(
#         partial(
#             run_training_job, 
#             dataset_dict=dataset_dict, 
#             train_data_key_in_dict=train_dataset_name, 
#             simulation_parent_id=sim_run_id_to_use, 
#             do_mlflow_model_logging=False,
#         ),
#         train_jobs
#     )

# All files have now been logged to MLFlow

To see how to process this data further look into the MLFlow loading and plotting notebook(s)